In [0]:
%run ./00-Config

In [0]:
# COMMAND ----------

# ==========================================
# IMPORTS
# ==========================================

from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.window import Window

from pyspark.sql.types import *

from datetime import datetime

In [0]:
# COMMAND ----------

# ==========================================
# CONFIGURAÇÕES
# ==========================================
# ALTERADO:
# tabelas agora usam variáveis globais

bronze_table = (
    f"{catalog_name}.bronze.openbrewery_bronze"
)

silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

control_table_name = (
    f"{catalog_name}.control.openbrewery_control"
)

In [0]:
# COMMAND ----------

# ==========================================
# OBTÉM ÚLTIMO TIMESTAMP PROCESSADO
# ==========================================

control_df = spark.sql(f"""

SELECT
    MAX(last_ingestion_timestamp) AS last_ts

FROM {control_table_name}

WHERE pipeline_name = 'openbrewery_silver'

""")

last_ts = control_df.collect()[0]["last_ts"]

print("=" * 60)
print(f"Último timestamp processado: {last_ts}")
print("=" * 60)

In [0]:
# COMMAND ----------

# ==========================================
# LEITURA DA BRONZE
# ==========================================

bronze_df = spark.read.table(
    bronze_table
)

In [0]:
# COMMAND ----------

# ==========================================
# PROCESSAMENTO INCREMENTAL
# ==========================================
# ALTERADO:
# processa apenas registros novos

if last_ts:

    bronze_df = bronze_df.filter(

        F.col("ingestion_timestamp")
        >
        F.lit(last_ts)

    )

In [0]:
# COMMAND ----------

# ==========================================
# REGRAS DE QUALIDADE
# ==========================================

silver_df = (

    bronze_df

    # ==========================================
    # ID OBRIGATÓRIO
    # ==========================================

    .filter(
        F.col("id").isNotNull()
    )

    # ==========================================
    # NAME OBRIGATÓRIO
    # ==========================================

    .filter(
        F.col("name").isNotNull()
    )

    # ==========================================
    # LATITUDE VÁLIDA
    # ==========================================

    .filter(

        (F.col("latitude").isNull())

        |

        (
            (F.col("latitude") >= -90)
            &
            (F.col("latitude") <= 90)
        )
    )

    # ==========================================
    # LONGITUDE VÁLIDA
    # ==========================================

    .filter(

        (F.col("longitude").isNull())

        |

        (
            (F.col("longitude") >= -180)
            &
            (F.col("longitude") <= 180)
        )
    )
)

In [0]:
# COMMAND ----------

# ==========================================
# PADRONIZAÇÕES
# ==========================================

silver_df = (

    silver_df

    .withColumn(
        "name",
        F.trim(F.col("name"))
    )

    .withColumn(
        "country",
        F.upper(F.col("country"))
    )

    .withColumn(
        "website_url",
        F.lower(F.col("website_url"))
    )

    .withColumn(

        "phone",

        F.regexp_replace(
            F.col("phone"),
            "[^0-9+]",
            ""
        )
    )
)

In [0]:
# COMMAND ----------

# ==========================================
# REMOVE DUPLICADOS MAIS RECENTES
# ==========================================

window_spec = (

    Window

    .partitionBy("id")

    .orderBy(
        F.col("ingestion_timestamp").desc()
    )
)

silver_df = (

    silver_df

    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )

    .filter(
        F.col("row_num") == 1
    )

    .drop("row_num")
)

In [0]:
# ==========================================
# COLUNAS SCD TYPE 2
# ==========================================
# ALTERADO:
# adicionada seleção explícita das colunas
# para evitar mismatch no append

silver_df = (

    silver_df

    .withColumn(
        "valid_from",
        F.current_timestamp()
    )

    .withColumn(
        "valid_to",
        F.lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        F.lit(True)
    )

    # ==========================================
    # ALTERADO:
    # mantém apenas colunas existentes
    # na tabela silver
    # ==========================================

    .select(

        "id",
        "name",
        "brewery_type",

        "address_1",
        "address_2",
        "address_3",

        "city",
        "state",
        "state_province",
        "postal_code",
        "country",

        "longitude",
        "latitude",

        "phone",
        "website_url",
        "street",

        "record_hash",

        "valid_from",
        "valid_to",
        "is_current",

        "ingestion_timestamp",
        "ingestion_date"
    )
)


In [0]:
# COMMAND ----------

# ==========================================
# REFERÊNCIA DELTA
# ==========================================

delta_table = DeltaTable.forName(
    spark,
    silver_table
)

In [0]:
# ==========================================
# REGISTROS ATUAIS
# ==========================================

current_df = (

    spark.read.table(silver_table)

    .filter(
        F.col("is_current") == True
    )
)

In [0]:
# COMMAND ----------

# ==========================================
# IDENTIFICA ALTERAÇÕES
# ==========================================

changes_df = (

    silver_df.alias("source")

    .join(

        current_df.alias("target"),

        on="id",

        how="left"
    )

    .filter(

        (
            F.col("target.id").isNull()
        )

        |

        (
            F.col("source.record_hash")
            !=
            F.col("target.record_hash")
        )
    )

    .select("source.*")
)

In [0]:
# COMMAND ----------

# ==========================================
# FECHA REGISTROS ANTIGOS
# ==========================================

(
    delta_table.alias("target")

    .merge(

        changes_df.alias("source"),

        """
        target.id = source.id
        AND target.is_current = true
        """
    )

    .whenMatchedUpdate(

        condition="""
            target.record_hash <> source.record_hash
        """,

        set={

            "is_current": "false",

            "valid_to": "current_timestamp()"
        }
    )

    .execute()
)

In [0]:
# COMMAND ----------

# ==========================================
# INSERE NOVOS REGISTROS
# ==========================================

(
    changes_df

    .select(
        *spark.read.table(silver_table).columns
    )

    .write

    .mode("append")

    .insertInto(silver_table)
)

In [0]:
# COMMAND ----------

# ==========================================
# CONTROLE INCREMENTAL
# ==========================================

records_processed = changes_df.count()

max_ts = (

    changes_df

    .agg(
        F.max("ingestion_timestamp").alias("max_ts")
    )

    .collect()[0]["max_ts"]
)

In [0]:
# COMMAND ----------

# ==========================================
# SALVA CONTROLE
# ==========================================

if max_ts:

    control_data = [

        (
            "openbrewery_silver",
            max_ts,
            datetime.utcnow(),
            records_processed
        )
    ]

    control_schema = StructType([

        StructField(
            "pipeline_name",
            StringType(),
            True
        ),

        StructField(
            "last_ingestion_timestamp",
            TimestampType(),
            True
        ),

        StructField(
            "processed_at",
            TimestampType(),
            True
        ),

        StructField(
            "records_processed",
            LongType(),
            True
        )
    ])

    control_insert_df = spark.createDataFrame(
        control_data,
        control_schema
    )

    (
        control_insert_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(control_table_name)
    )

In [0]:

# COMMAND ----------

# ==========================================
# FINALIZAÇÃO
# ==========================================

print("=" * 60)

print("SILVER PROCESSADA COM SUCESSO")

print(f"Tabela: {silver_table}")

print(f"Registros processados: {records_processed}")

print(f"Último timestamp processado: {max_ts}")

print("=" * 60)